In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import time

# fix an issue with the method of running in the notebook
# I probably just need to look into python packaging more
sys.path.append(os.path.join(os.getcwd(), "scripts"))

import numpy as np

import tensorflow as tf
from tensorflow.keras import Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

from scripts.utils import Config
from scripts.utils.tf.dataloaders import AlmLoader
from scripts.utils.tf.plots import plot_histogram, plot_metrics, plot_predictions
from scripts.utils.tf.callbacks import TimedLoggingCallback, WarmupLearningRate

from mlpng.attn_alm import alm_model

In [ ]:
s = Config("settings/ul_nn_128_large.json")

MAX_EPOCHS = 300
BATCH_SIZE = 32

# just some info for the model name
timestamp = int(time.time())
model_settings = {
    "dropout_rate": 0.3,
    "name": f"tester",
}

data_loader_args = {
    "shuffle": True,
    "seed": None,
    "batch_size": BATCH_SIZE,
    "cache": True,
    "shuffle_buffer": 1000,
}

# additional metrics we are intrested in
metrics = ["mean_absolute_error"]

In [ ]:
lr_schedule = WarmupLearningRate(
    warmup_learning_rate=1e-8,  # start small
    warmup_steps=1e5,
    warmup_scale=50,
    warmup_scale_steps=1,
    warmed_learning_rate=1e-3,
    decay_steps=1000,
    decay_rate=0.95,
    staircase=True,
)

In [ ]:
strategy = tf.distribute.MirroredStrategy()
print(strategy.num_replicas_in_sync)
# strategy = tf.distribute.OneDeviceStrategy(device="/CPU:0")
num_gpus = strategy.num_replicas_in_sync

data_loader = AlmLoader(
    s.alm_file_complete, num_replicas=num_gpus, **data_loader_args
)
train_dataset, test_dataset, val_dataset = data_loader.get_split(0.8, 0.1, 0.1)

with strategy.scope():
    opt = Adam(learning_rate=lr_schedule)

    model = alm_model(Input(data_loader.shape), **model_settings)
    # model = simple_transformer(Input(data_loader.shape), **model_settings)

    model.compile(optimizer=opt, loss=tf.keras.losses.mse, metrics=metrics)

In [ ]:
model.summary()

In [ ]:
callbacks = [
    # We use earlystoping to prevent overfitting
    EarlyStopping(
        monitor="val_loss",
        patience=20,
        verbose=1,
        restore_best_weights=True,
        start_from_epoch=50,
    ),
    # TimedLoggingCallback(print_frequency=60),
    # TensorBoard(
    #     log_dir=f"{s.tb_dir}/{model_settings['name']}",
    #     histogram_freq=1,
    # ),
]

In [ ]:
if False:
    import wandb
    from wandb.keras import WandbMetricsLogger

    # wandb.tensorboard.patch(root_logdir=s.tb_dir)

    wandb.init(
        project="mlpng",
        tags=["tester", "dev"],
        config=s.settings | model_settings,
        dir="data",
        sync_tensorboard=True,
    )

    # Add the wandb logger to the callbacks, so it is used
    callbacks.append(WandbMetricsLogger())

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# Lets plot the predictions from the unseen test set
y_pred = model.predict(test_dataset, verbose=0).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_dataset])

In [ ]:
# Plot the loss curves and metrics
plot_metrics(history, metrics=["loss"] + metrics)
plot_predictions(y_test, y_pred)
plot_histogram(y_test, y_pred)